# P3 — Đánh giá cấp sự kiện (Thuật toán 2.2, 3.1), Bảng 2.3 bổ sung, Bảng 3.14

**Không cần GPU, không huấn luyện.** Đầu vào:
1. `exp099_window_predictions.csv` — xuất từ notebook 01 (ô T6).
2. Thư mục dữ liệu ver3 có `upfall_train.npz`, `upfall_val.npz`, `upfall_test.npz`.

**Nguyên tắc (theo góp ý của cô):** mọi tham số của bộ quản lý cảnh báo được chọn trên **S11–S12**,
sau đó áp lên **S13–S17 đúng một lần**.

**Thời gian:** cần tần số khung hình `FPS` (f̂_fps, công thức 2.1) để đổi chỉ số khung sang giây.
Trong lúc chưa tính được f̂_fps từ timestamp, notebook dùng 18 khung/giây theo mô tả của UP-Fall —
**phải thay bằng f̂_fps thật trước khi đưa số vào khóa luận.**


In [30]:
%%writefile alert_manager.py
"""
alert_manager.py — Thuật toán 2.2: Bộ quản lý cảnh báo bốn trạng thái.

Dùng CHUNG một mã cho:
  * đánh giá phát lại trên UP-Fall (Thuật toán 3.1, notebook 02)
  * hệ thống chạy thời gian thực trên thiết bị (realtime_pipeline.py)

Trạng thái: NORMAL (BÌNH_THƯỜNG) -> SUSPECT (NGHI_NGỜ) -> CONFIRMED (XÁC_NHẬN) -> COOLDOWN (CHỜ)
"""
from dataclasses import dataclass, field

NORMAL, SUSPECT, CONFIRMED, COOLDOWN = 'NORMAL', 'SUSPECT', 'CONFIRMED', 'COOLDOWN'


@dataclass
class AlertParams:
    tau: float                 # ngưỡng phân loại τ
    theta_lie: float = 60.0    # ngưỡng góc nằm θ_lie (độ); 0 = bỏ bước xác nhận tư thế
    t_post: float = 3.0        # T_post (giây): thời gian tối đa chờ xác nhận
    t_cool: float = 30.0       # T_c (giây): thời gian chờ sau mỗi cảnh báo


@dataclass
class AlertManager:
    params: AlertParams
    state: str = NORMAL
    t_s: float = 0.0           # thời điểm bắt đầu nghi ngờ
    p_s: float = 0.0           # xác suất lúc bắt đầu nghi ngờ
    t_alert: float = float('-inf')
    alerts: list = field(default_factory=list)   # danh sách (t_alert, p_s)

    def reset(self):
        self.state = NORMAL
        self.t_s = 0.0
        self.p_s = 0.0
        self.t_alert = float('-inf')
        self.alerts = []

    def update(self, t_j, p_j, theta_j):
        """
        Xử lý một cửa sổ mới (bước 2 của Thuật toán 2.2).
        t_j: thời điểm kết thúc cửa sổ (giây); p_j: xác suất Fall; theta_j: θ̄_j (độ).
        Trả về True nếu phát cảnh báo tại cửa sổ này.
        """
        P = self.params
        # 2.1
        if self.state == COOLDOWN and (t_j - self.t_alert) >= P.t_cool:
            self.state = NORMAL
        # 2.2
        if self.state == NORMAL and p_j >= P.tau:
            self.state = SUSPECT
            self.t_s = t_j
            self.p_s = p_j
        # 2.3
        if self.state == SUSPECT:
            if theta_j >= P.theta_lie:
                self.state = CONFIRMED
            elif (t_j - self.t_s) > P.t_post:
                self.state = NORMAL
        # 2.4
        if self.state == CONFIRMED:
            self.t_alert = t_j
            self.alerts.append((t_j, self.p_s))
            self.state = COOLDOWN
            return True
        return False


def run_alert_manager(times, probs, thetas, params):
    """Chạy Thuật toán 2.2 trên một dãy cửa sổ đã sắp theo thời gian; trả về danh sách cảnh báo."""
    am = AlertManager(params)
    for t, p, th in zip(times, probs, thetas):
        am.update(float(t), float(p), float(th))
    return am.alerts


def run_every_window(times, probs, tau):
    """Chính sách so sánh: mỗi cửa sổ có p >= τ là một cảnh báo."""
    return [(float(t), float(p)) for t, p in zip(times, probs) if p >= tau]


Overwriting alert_manager.py


In [31]:
%%writefile event_eval.py
"""
event_eval.py — Thuật toán 3.1 (đánh giá cấp sự kiện bằng phát lại) và các tiện ích thống kê
dùng cho Bảng 2.3, Bảng 3.9, Bảng 3.14.

Không cần TensorFlow. Đầu vào là bảng xác suất từng cửa sổ (do notebook 01 xuất ra) và
siêu dữ liệu cửa sổ trong upfall_*.npz (subject, activity, trial, start_frame, end_frame, fall_ratio).
"""
import numpy as np
import pandas as pd

from alert_manager import AlertParams, run_alert_manager, run_every_window

WIN = 30
FALL_ACTIVITIES = {1, 2, 3, 4, 5}   # UP-Fall: hoạt động 1–5 là té ngã


# ---------------------------------------------------------------------
# 1) Khôi phục pha ngã [a, b] (chỉ số khung) của mỗi lượt từ fall_ratio của các cửa sổ
# ---------------------------------------------------------------------
def reconstruct_fall_interval(starts, ends, n_fall):
    """
    Tìm đoạn khung liên tiếp [a, b] sao cho số khung ngã trong mỗi cửa sổ
    |[s, e] ∩ [a, b]| khớp nhất với n_fall (= round(fall_ratio * 30)).
    Trả về (a, b, residual). residual = 0 nghĩa là khớp chính xác.
    """
    starts = np.asarray(starts, int)
    ends = np.asarray(ends, int)
    n_fall = np.asarray(n_fall, int)
    pos = n_fall > 0
    if not pos.any():
        return None, None, None
    lo = int(starts[pos].min())
    hi = int(ends[pos].max())
    best = (None, None, 10 ** 9)
    for a in range(lo, hi + 1):
        for b in range(a, hi + 1):
            ov = np.clip(np.minimum(ends, b) - np.maximum(starts, a) + 1, 0, None)
            r = int(np.abs(ov - n_fall).sum())
            if r < best[2]:
                best = (a, b, r)
                if r == 0:
                    break
        if best[2] == 0:
            break
    return best


def trial_table(meta):
    """
    meta: DataFrame có cột split, subject, activity, trial, start_frame, end_frame, fall_ratio, y.
    Trả về bảng mỗi lượt: số cửa sổ, số cửa sổ Fall, pha ngã [a, b], ℓ (khung), residual.
    """
    rows = []
    for key, g in meta.groupby(['split', 'subject', 'activity', 'trial']):
        g = g.sort_values('start_frame')
        split, s, act, tr = key
        n_fall_frames = np.rint(g['fall_ratio'].to_numpy() * WIN).astype(int)
        row = dict(split=split, subject=int(s), activity=int(act), trial=int(tr),
                   n_windows=len(g), n_fall_windows=int(g['y'].sum()),
                   last_frame=int(g['end_frame'].max()),
                   is_fall_trial=int(act) in FALL_ACTIVITIES)
        if row['is_fall_trial']:
            a, b, r = reconstruct_fall_interval(g['start_frame'], g['end_frame'], n_fall_frames)
            row.update(fall_start=a, fall_end=b,
                       ell=(None if a is None else b - a + 1), residual=r)
        rows.append(row)
    return pd.DataFrame(rows)


def table_2_3_extra(trials):
    """Các cột bổ sung cho Bảng 2.3 theo góp ý của cô."""
    out = []
    for split, g in trials[trials.is_fall_trial].groupby('split'):
        ell = g['ell'].dropna()
        out.append(dict(
            split=split,
            fall_trials_with_windows=len(g),
            fall_windows=int(g['n_fall_windows'].sum()),
            fall_windows_per_trial=round(g['n_fall_windows'].mean(), 2),
            ell_mean=round(ell.mean(), 1), ell_min=int(ell.min()), ell_max=int(ell.max()),
            trials_without_fall_window=int((g['n_fall_windows'] == 0).sum()),
            trials_exact_reconstruction=int((g['residual'] == 0).sum()),
        ))
    return pd.DataFrame(out)


# ---------------------------------------------------------------------
# 2) Thuật toán 3.1 — phát lại theo từng lượt ghi hình
# ---------------------------------------------------------------------
def event_level_eval(pred, trials, fps, policy, params, delta=5.0):
    """
    pred: DataFrame các cửa sổ của MỘT tập (vd. test), có cột subject, activity, trial,
          start_frame, end_frame, p, theta_deg.
    trials: kết quả trial_table() cho cùng tập.
    policy: 'every_window' hoặc 'alert_manager'.
    Trả về dict các chỉ số (1.25)–(1.27) và bảng chi tiết từng lượt.
    """
    n_fall = n_det = n_false = 0
    t_adl = 0.0
    delays = []
    detail = []
    tinfo = trials.set_index(['subject', 'activity', 'trial'])
    for key, g in pred.groupby(['subject', 'activity', 'trial']):
        g = g.sort_values('start_frame')
        times = (g['end_frame'].to_numpy() + 1) / fps          # t_j: thời điểm kết thúc cửa sổ
        probs = g['p'].to_numpy()
        thetas = g['theta_deg'].to_numpy()
        if policy == 'every_window':
            alerts = run_every_window(times, probs, params.tau)
        else:
            alerts = run_alert_manager(times, probs, thetas, params)
        info = tinfo.loc[key]
        D_r = (int(info['last_frame']) + 1) / fps
        rec = dict(subject=key[0], activity=key[1], trial=key[2], n_alerts=len(alerts))
        if bool(info['is_fall_trial']) and info['fall_start'] is not None and not pd.isna(info['fall_start']):
            n_fall += 1
            t_start = float(info['fall_start']) / fps
            t_end = (float(info['fall_end']) + 1) / fps
            matched = [a for a in alerts if t_start <= a[0] <= t_end + delta]
            if matched:
                n_det += 1
                d = min(a[0] for a in matched) - t_start
                delays.append(d)
                rec['delay_s'] = d
            n_false += len(alerts) - len(matched)
            t_adl += max(D_r - (t_end + delta - t_start), 0.0)
            rec.update(fall_trial=True, detected=bool(matched))
        else:
            n_false += len(alerts)
            t_adl += D_r
            rec.update(fall_trial=False, detected=None)
        detail.append(rec)
    res = dict(
        n_fall=n_fall, n_detected=n_det,
        recall_ev=n_det / max(n_fall, 1),
        delay_median_s=float(np.median(delays)) if delays else float('nan'),
        delay_max_s=float(np.max(delays)) if delays else float('nan'),
        n_false_alerts=n_false, t_adl_h=t_adl / 3600.0,
        fa_per_h=n_false / max(t_adl / 3600.0, 1e-9),
    )
    return res, pd.DataFrame(detail)


# ---------------------------------------------------------------------
# 3) Chỉ số cấp cửa sổ dùng chung
# ---------------------------------------------------------------------
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


Overwriting event_eval.py


In [32]:
# ============================================================
# 1. CẤU HÌNH
# ============================================================
import json, numpy as np, pandas as pd
from pathlib import Path
from alert_manager import AlertParams
from event_eval import trial_table, table_2_3_extra, event_level_eval, wilson_ci

PRED_CSV = Path('/kaggle/working/KLTN_EXP_099/exp099_window_predictions.csv')   # sửa nếu cần
CONFIG_JSON = Path('/kaggle/working/KLTN_EXP_099/thesis_chosen_config.json')
NPZ_DIR = Path('/kaggle/input/datasets/anhoa1/upfall-camera2-processing')  # thư mục chứa upfall_*.npz
OUT = Path('/kaggle/working/P3_outputs'); OUT.mkdir(parents=True, exist_ok=True)

FPS = 18.0            # TODO: thay bằng f̂_fps đo từ timestamp (công thức 2.1)
DELTA = 5.0           # δ (giây): dung sai sau khi kết thúc ngã
N_FALL_TOTAL_TEST = 75  # số lượt ngã của S13–S17 trong UP-Fall (5 người × 5 kiểu ngã × 3 lượt)
T_COOL = 30.0; T_POST = 3.0

pred = pd.read_csv(PRED_CSV)
cfg = json.load(open(CONFIG_JSON))
TAU = float(cfg['threshold_tau'])
print('τ của EXP_099 =', TAU, '| α =', cfg['alpha_pmg'])
print(pred.groupby('split').size())


τ của EXP_099 = 0.7840000000000003 | α = 0.525
split
test    4786
val     2046
dtype: int64


In [33]:
# ============================================================
# 2. BẢNG 2.3 BỔ SUNG: ℓ (độ dài pha ngã), số cửa sổ Fall mỗi lượt,
#    số lượt ngã không sinh ra cửa sổ Fall nào
#    ℓ được khôi phục chính xác từ fall_ratio của các cửa sổ chồng lấn (residual = 0)
# ============================================================
metas = []
for split in ['train', 'val', 'test']:
    d = np.load(NPZ_DIR/f'upfall_{split}.npz', allow_pickle=True)
    m = pd.DataFrame({k: np.asarray(d[k]).reshape(-1) for k in
                      ['subject', 'activity', 'trial', 'start_frame', 'end_frame', 'fall_ratio', 'y']})
    m['split'] = split
    metas.append(m)
meta_all = pd.concat(metas, ignore_index=True)
trials_all = trial_table(meta_all)
t23 = table_2_3_extra(trials_all)
display(t23)
t23.to_csv(OUT/'table_2_3_extra.csv', index=False)
trials_all.to_csv(OUT/'trials_fall_phase.csv', index=False)
print('ℓ tính bằng khung; đổi sang giây: ℓ / FPS. Các lượt có residual ≠ 0:',
      int((trials_all['residual'].fillna(0) != 0).sum()))


,split,fall_trials_with_windows,fall_windows,fall_windows_per_trial,ell_mean,ell_min,ell_max,trials_without_fall_window,trials_exact_reconstruction
0,test,72,100,1.39,26.2,8,59,9,71
1,train,149,314,2.11,33.9,4,66,18,147
2,val,30,56,1.87,29.2,13,48,3,30


ℓ tính bằng khung; đổi sang giây: ℓ / FPS. Các lượt có residual ≠ 0: 0


In [34]:
# ============================================================
# 3. BẢNG 3.14 — các cửa sổ phân loại sai của EXP_099 trên S11–S12
#    Sau khi chạy: mở video gốc từng dòng để điền cột "Nhận xét sau khi xem video".
# ============================================================
ACT_NAMES = {1: 'Ngã về trước (chống tay)', 2: 'Ngã về trước (đầu gối)', 3: 'Ngã ra sau',
             4: 'Ngã sang bên', 5: 'Ngã khi ngồi ghế', 6: 'Đi bộ', 7: 'Đứng', 8: 'Ngồi',
             9: 'Nhặt đồ', 10: 'Nhảy', 11: 'Nằm'}
val = pred[pred.split == 'val'].copy()
err = val[val.pred_exp099 != val.y].copy()
err['Loại'] = np.where(err.y == 1, 'FN', 'FP')
err = err.sort_values(['Loại', 'subject', 'activity', 'trial', 'start_frame'])
t314 = pd.DataFrame({
    'STT': np.arange(1, len(err)+1), 'Loại': err['Loại'].values,
    'Người': ['S%d' % s for s in err.subject], 'Hoạt động': [f'{a} – {ACT_NAMES.get(a, "")}' for a in err.activity],
    'Lượt': err.trial.values, 'Khung bắt đầu': err.start_frame.values,
    'φj': err.fall_ratio.round(3).values, 'p': err.p.round(3).values,
    'θ̄j (độ)': err.theta_deg.round(1).values, 'Nhận xét sau khi xem video': '',
})
display(t314)
t314.to_csv(OUT/'table_3_14_val_errors.csv', index=False)
fn = err[err.y == 1]
print(f'FN: {len(fn)} | trong đó φj trong [0,5; 0,6]: {int(((fn.fall_ratio >= 0.5) & (fn.fall_ratio <= 0.6)).sum())} '
      '(nếu nhiều → lỗi nằm ở quy tắc gán nhãn (2.2), không phải mô hình)')
print('FP theo hoạt động:', err[err.y == 0].activity.map(ACT_NAMES).value_counts().to_dict())


,STT,Loại,Người,Hoạt động,Lượt,Khung bắt đầu,φj,p,θ̄j (độ),Nhận xét sau khi xem video
0,1,FN,S11,3 – Ngã ra sau,1,15,0.533,0.005,6.9,
1,2,FN,S11,4 – Ngã sang bên,3,0,0.600,0.776,8.5,
2,3,FN,S11,5 – Ngã khi ngồi ghế,2,0,0.500,0.081,0.8,
3,4,FN,S12,1 – Ngã về trước (chống tay),2,15,0.667,0.683,4.0,
4,5,FN,S12,3 – Ngã ra sau,1,0,0.600,0.071,9.8,
5,6,FN,S12,5 – Ngã khi ngồi ghế,2,15,1.000,0.733,15.1,
6,7,FN,S12,5 – Ngã khi ngồi ghế,3,0,0.533,0.700,14.2,
7,8,FN,S12,5 – Ngã khi ngồi ghế,3,15,1.000,0.074,4.0,
8,9,FN,S12,5 – Ngã khi ngồi ghế,3,30,0.600,0.040,9.4,
9,10,FP,S11,1 – Ngã về trước (chống tay),1,15,0.467,0.897,10.6,


FN: 9 | trong đó φj trong [0,5; 0,6]: 6 (nếu nhiều → lỗi nằm ở quy tắc gán nhãn (2.2), không phải mô hình)
FP theo hoạt động: {'Ngã ra sau': 2, 'Ngã về trước (đầu gối)': 2, 'Ngã về trước (chống tay)': 1, 'Ngã khi ngồi ghế': 1}


In [35]:
# ============================================================
# 4. ƯỚC LƯỢNG (2.10): số báo động giả mỗi giờ nếu mỗi cửa sổ vượt ngưỡng là một cảnh báo
# ============================================================
T_STEP = 15
r_fp = ((val.p >= TAU) & (val.y == 0)).sum() / max((val.y == 0).sum(), 1)
fa_h_est = 3600 * FPS / T_STEP * r_fp
print(f'r_FP (kiểm chứng) = {r_fp:.4f} → FA/h ước lượng ≈ {fa_h_est:.1f} (với f̂_fps = {FPS})')


r_FP (kiểm chứng) = 0.0030 → FA/h ước lượng ≈ 13.0 (với f̂_fps = 18.0)


In [36]:
# ============================================================
# 5. CHỌN THAM SỐ BỘ QUẢN LÝ CẢNH BÁO TRÊN S11–S12 (KHÔNG dùng tập kiểm thử)
#   * θ_lie: lưới 0–60°; tiêu chí: Recall_ev lớn nhất, rồi FA/h nhỏ nhất
#   * τ_low: τ lớn nhất có Recall cấp cửa sổ trên S11–S12 ≥ 95% (gợi ý của cô cho Bảng 3.9)
# ============================================================
trials_val = trials_all[trials_all.split == 'val']
grid = []
for th in [0, 10, 15, 20, 25, 30, 40, 50, 60]:
    r, _ = event_level_eval(val, trials_val, FPS, 'alert_manager',
                            AlertParams(TAU, theta_lie=th, t_post=T_POST, t_cool=T_COOL), DELTA)
    grid.append({'theta_lie': th, **r})
grid = pd.DataFrame(grid)
display(grid.round(3))
best = grid.sort_values(['recall_ev', 'fa_per_h'], ascending=[False, True]).iloc[0]
THETA_STAR = float(best['theta_lie'])
print('θ_lie chọn trên S11–S12 =', THETA_STAR)

taus = np.round(np.arange(0.05, 0.991, 0.001), 3)
ok = [t for t in taus if ((val.p >= t) & (val.y == 1)).sum() / max((val.y == 1).sum(), 1) >= 0.95]
TAU_LOW = float(max(ok)) if ok else float(taus[0])
print('τ_low (Recall cửa sổ kiểm chứng ≥ 95%) =', TAU_LOW)
grid.to_csv(OUT/'alert_param_selection_val.csv', index=False)
json.dump({'FPS': FPS, 'DELTA': DELTA, 'TAU': TAU, 'THETA_STAR': THETA_STAR, 'TAU_LOW': TAU_LOW,
           'T_COOL': T_COOL, 'T_POST': T_POST}, open(OUT/'alert_params_chosen.json', 'w'), indent=2)


,theta_lie,n_fall,n_detected,recall_ev,delay_median_s,delay_max_s,n_false_alerts,t_adl_h,fa_per_h
0,0,30,28,0.933,1.417,1.833,0,0.493,0.0
1,10,30,19,0.633,1.500,4.111,0,0.493,0.0
2,15,30,12,0.400,1.444,3.889,0,0.493,0.0
3,20,30,5,0.167,1.444,1.667,0,0.493,0.0
4,25,30,4,0.133,1.389,1.444,0,0.493,0.0
5,30,30,3,0.100,1.444,1.444,0,0.493,0.0
6,40,30,1,0.033,1.444,1.444,0,0.493,0.0
7,50,30,0,0.000,NaN,NaN,0,0.493,0.0
8,60,30,0,0.000,NaN,NaN,0,0.493,0.0


θ_lie chọn trên S11–S12 = 0.0
τ_low (Recall cửa sổ kiểm chứng ≥ 95%) = 0.07


In [37]:
# ============================================================
# 6. BẢNG 3.9 — đánh giá cấp sự kiện trên S13–S17 (chạy MỘT lần với tham số đã chốt ở bước 5)
# ============================================================
test = pred[pred.split == 'test'].copy()
trials_test = trials_all[trials_all.split == 'test']
policies = [
    ('Mỗi cửa sổ có p ≥ τ là một cảnh báo', 'every_window', AlertParams(TAU)),
    (f'Thuật toán 2.2, chỉ thời gian chờ (θ_lie = 0°, T_c = {T_COOL:.0f} s)', 'alert_manager',
     AlertParams(TAU, theta_lie=0, t_post=T_POST, t_cool=T_COOL)),
    (f'Thuật toán 2.2 đầy đủ, thiết kế ban đầu (θ_lie = 60°)', 'alert_manager',
     AlertParams(TAU, theta_lie=60, t_post=T_POST, t_cool=T_COOL)),
    (f'Thuật toán 2.2, θ_lie = {THETA_STAR:.0f}° chọn trên S11–S12', 'alert_manager',
     AlertParams(TAU, theta_lie=THETA_STAR, t_post=T_POST, t_cool=T_COOL)),
    (f'τ thấp = {TAU_LOW} + Thuật toán 2.2 (θ_lie = {THETA_STAR:.0f}°)', 'alert_manager',
     AlertParams(TAU_LOW, theta_lie=THETA_STAR, t_post=T_POST, t_cool=T_COOL)),
]
rows, details = [], {}
for name, pol, P in policies:
    r, det = event_level_eval(test, trials_test, FPS, pol, P, DELTA)
    lo, hi = wilson_ci(r['n_detected'], N_FALL_TOTAL_TEST)
    rows.append({
        'Chính sách cảnh báo': name,
        'Cú ngã phát hiện': f"{r['n_detected']}/{N_FALL_TOTAL_TEST}",
        'Recall_ev (%) /75': 100*r['n_detected']/N_FALL_TOTAL_TEST,
        'Recall_ev 95% CI': f'[{100*lo:.1f}; {100*hi:.1f}]',
        f"Recall_ev (%) /{r['n_fall']} lượt có pha ngã trong dữ liệu": 100*r['recall_ev'],
        'Độ trễ trung vị (s)': r['delay_median_s'], 'Độ trễ lớn nhất (s)': r['delay_max_s'],
        'Số báo động giả': r['n_false_alerts'], 'FA/h': r['fa_per_h'], 'T_ADL (giờ)': r['t_adl_h'],
    })
    details[name] = det
table_3_9 = pd.DataFrame(rows)
display(table_3_9.round(2))
table_3_9.to_csv(OUT/'table_3_9_event_level_test.csv', index=False)
missing = N_FALL_TOTAL_TEST - int(trials_test.is_fall_trial.sum())
print(f'Ghi chú: {missing} lượt ngã của S13–S17 không có cửa sổ nào sau tiền xử lý → tính là bỏ sót trong cột "/75". '
      f'{int((trials_test[trials_test.is_fall_trial].n_fall_windows == 0).sum())} lượt có cửa sổ nhưng không có cửa sổ Fall nào.')
print('T_ADL chỉ khoảng 1 giờ → FA/h có độ bất định lớn; nêu khi bình luận.')


,Chính sách cảnh báo,Cú ngã phát hiện,Recall_ev (%) /75,Recall_ev 95% CI,Recall_ev (%) /71 lượt có pha ngã trong dữ liệu,Độ trễ trung vị (s),Độ trễ lớn nhất (s),Số báo động giả,FA/h,T_ADL (giờ)
0,Mỗi cửa sổ có p ≥ τ là một cảnh báo,52/75,69.33,[58.2; 78.6],73.24,1.50,2.17,4,3.52,1.14
1,"Thuật toán 2.2, chỉ thời gian chờ (θ_lie = 0°,...",52/75,69.33,[58.2; 78.6],73.24,1.50,2.17,4,3.52,1.14
2,"Thuật toán 2.2 đầy đủ, thiết kế ban đầu (θ_lie...",0/75,0.00,[0.0; 4.9],0.00,NaN,NaN,0,0.00,1.14
3,"Thuật toán 2.2, θ_lie = 0° chọn trên S11–S12",52/75,69.33,[58.2; 78.6],73.24,1.50,2.17,4,3.52,1.14
4,τ thấp = 0.07 + Thuật toán 2.2 (θ_lie = 0°),66/75,88.00,[78.7; 93.6],92.96,1.11,1.83,38,33.42,1.14


Ghi chú: 3 lượt ngã của S13–S17 không có cửa sổ nào sau tiền xử lý → tính là bỏ sót trong cột "/75". 9 lượt có cửa sổ nhưng không có cửa sổ Fall nào.
T_ADL chỉ khoảng 1 giờ → FA/h có độ bất định lớn; nêu khi bình luận.


In [38]:
# ============================================================
# 7. PHÂN BỐ θ̄ SAU KHI NGÃ — kiểm tra giả định "nằm ngang" của bước xác nhận tư thế
#    (camera 2 nhìn phía trước: ngã về trước/ra sau làm thân người ngắn lại theo trục dọc
#     chứ không nghiêng sang ngang trong mặt phẳng ảnh)
# ============================================================
rows = []
for split, df in [('val', val), ('test', test)]:
    tt = trials_all[trials_all.split == split].set_index(['subject', 'activity', 'trial'])
    for key, g in df.groupby(['subject', 'activity', 'trial']):
        info = tt.loc[key]
        if info.is_fall_trial and not pd.isna(info.fall_start):
            w = g[(g.end_frame >= info.fall_start) & (g.start_frame <= info.fall_end + DELTA*FPS)]
            rows.append({'split': split, 'nhóm': f'Ngã – HĐ {key[1]}', 'θ̄ lớn nhất': w.theta_deg.max()})
        elif not info.is_fall_trial:
            rows.append({'split': split, 'nhóm': f'ADL – HĐ {key[1]}', 'θ̄ lớn nhất': g.theta_deg.max()})
theta_df = pd.DataFrame(rows)
theta_summary = theta_df.groupby(['split', 'nhóm'])['θ̄ lớn nhất'].agg(['count', 'median', 'min', 'max']).round(1)
display(theta_summary)
theta_summary.to_csv(OUT/'theta_after_fall_summary.csv')

import shutil
print('Tải về:', shutil.make_archive('/kaggle/working/P3_outputs', 'zip', OUT))


count  median   min   max
split nhóm                                  
test  ADL – HĐ 10     15     6.1   3.9   9.2
      ADL – HĐ 11     15    15.6   7.5  34.3
      ADL – HĐ 6      15    17.1   7.7  49.5
      ADL – HĐ 7      15     6.5   2.6  15.0
      ADL – HĐ 8      15    14.9   6.1  37.8
      ADL – HĐ 9      15    26.6   5.3  86.2
      Ngã – HĐ 1      15    13.4   2.5  40.8
      Ngã – HĐ 2      15    11.5   0.8  37.5
      Ngã – HĐ 3      14    13.0   1.9  32.0
      Ngã – HĐ 4      15    14.8   6.8  54.4
      Ngã – HĐ 5      12    22.9   4.3  45.8
val   ADL – HĐ 10      6     9.0   8.2  12.2
      ADL – HĐ 11      6    27.3  13.0  32.4
      ADL – HĐ 6       6    19.5   8.3  35.5
      ADL – HĐ 7       6     6.0   4.1   6.4
      ADL – HĐ 8       6    17.8  17.3  23.3
      ADL – HĐ 9       6    42.6   7.9  51.7
      Ngã – HĐ 1       6    13.6  12.1  28.2
      Ngã – HĐ 2       6     7.8   1.8  17.8
      Ngã – HĐ 3       6    17.5   7.9  38.4
      Ngã – HĐ 4       6    10.8   4.2  50.0
      Ngã – HĐ 5       6    18.9  13.1  30.9

Tải về: /kaggle/working/P3_outputs.zip
